In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import time
import re

start_time = time.time()
NUM_RECORDS = 500000
print(f"กำลังสุ่มสร้างข้อมูล IPD ภาษามนุษย์จำนวน {NUM_RECORDS:,} เรคคอร์ด...")

# ==========================================
# 1. โครงสร้างประโยคแบบธรรมชาติ (Natural Templates)
# ==========================================

# --- คำชม (Positive) ---
pos_intros = ["ประทับใจมาก", "ขอบคุณทีมงานครับ", "รู้สึกอบอุ่นใจ", "ดีเยี่ยมเลย", "อยากมาชื่นชม", ""]
pos_subjects = ["คุณหมอ", "พยาบาลวอร์ดนี้", "พี่ๆ ผู้ช่วย", "เจ้าหน้าที่เวรดึก", "แม่บ้าน", "ทีมแพทย์"]
pos_actions = ["ดูแลดีดุจญาติมิตร", "เอาใจใส่สุดๆ", "พูดจาไพเราะมาก", "อธิบายอาการชัดเจนเข้าใจง่าย", "บริการรวดเร็วทันใจ", "มือเบามากตอนเจาะเลือด"]
pos_details = ["เดินมาเช็คอาการตลอดเลย", "กดออดเรียกตอนดึกก็มาไวไม่เหวี่ยง", "ห้องพักสะอาดมากแอร์เย็นฉ่ำ", "อาหารรพ.อร่อยกว่าที่คิดไว้เยอะ", "ได้ห้องวิวดีพักผ่อนสบาย"]
pos_endings = ["ขอชื่นชมจากใจครับ", "เป็นกำลังใจให้คนทำงานนะคะ", "รักษามาตรฐานดีๆ แบบนี้ไว้นะ", "เยี่ยมจริงๆ ค่ะ", "ประทับใจสุดๆ", ""]

# --- คำติ/บ่น (Negative) ---
neg_intros = ["ขอติเรื่องนึงนะคะ", "ไม่ประทับใจเลย", "อยากให้ปรับปรุงด่วน", "แย่มากๆ", "แอบผิดหวัง", "ขอระบายหน่อย"]
neg_subjects = ["พยาบาลบางคน", "ระบบการจัดการ", "ห้องน้ำในห้องพัก", "ที่จอดรถของญาติ", "อาหารผู้ป่วย", "แอร์ในห้อง","พยาบาลแผนกนี้"]
neg_actions = ["พูดจาไม่ค่อยดีเลย", "รอนานเกินไปไหม", "มีกลิ่นเหม็นอับทะลุออกมา", "หายากมากวนหาเป็นชั่วโมง", "จืดสนิทจนกินไม่ลง", "เสียงดังและไม่ค่อยเย็น"]
neg_details = ["หน้าบึ้งเหมือนโกรธใครมา", "ขั้นตอนซับซ้อนงงไปหมด", "เตียงแข็งปวดหลังสุดๆ นอนไม่หลับ", "หมอมาตรวจแป๊บเดียวถามอาการไม่ทัน", "กว่าจะได้ทำเรื่องออกจากรพ.ก็ปาไปครึ่งวัน"]
neg_endings = ["ฝากผู้บริหารลงมาดูด้วยครับ", "ควรแก้ไขด่วนเลยนะ", "เสียความรู้สึกมาก", "เหนื่อยใจจริงๆ", "ไม่โอเคเลยค่ะ", ""]

# ==========================================
# 2. ฟังก์ชันเติมความเป็นมนุษย์ (Humanizer & Augmentation)
# ==========================================
def humanize_text(text, sentiment):
    # 1. สุ่มแปลงคำให้วิบัติ/ลากเสียง (Typo & Slang Injection)
    if random.random() < 0.3:
        text = text.replace("มาก", random.choice(["ม้ากกก", "มากๆ", "ม๊าก"]))
    if random.random() < 0.2:
        text = text.replace("เลย", random.choice(["เลยยย", "เรย", "เลยจ้า"]))
    if random.random() < 0.15:
        text = text.replace("พยาบาล", "พยาบาน")

    # 2. สุ่มเติมหางเสียง
    if random.random() < 0.5:
        tail = random.choice([" ครับ", " ค่ะ", " นะคะ", " ครับผม", " จ้า", " คับ"])
        text += tail

    # 3. สุ่มใส่อีโมจิ
    if random.random() < 0.4:
        if sentiment == 'positive':
            text += random.choice([" 😊", " ❤️", " 🙏", " 👍", " 🥰"])
        else:
            text += random.choice([" 😡", " 🙄", " 😤", " 👎", " 😩", " 😓"])

    return text.strip()

def generate_natural_comment(sentiment):
    if sentiment == 'positive':
        components = [random.choice(pos_intros), random.choice(pos_subjects), random.choice(pos_actions), random.choice(pos_details), random.choice(pos_endings)]
    else:
        components = [random.choice(neg_intros), random.choice(neg_subjects), random.choice(neg_actions), random.choice(neg_details), random.choice(neg_endings)]

    # เอาส่วนประกอบมารวมกัน (ข้ามอันที่เป็นค่าว่าง)
    raw_text = " ".join([c for c in components if c])

    # นำไปผ่านฟังก์ชันเติมความเป็นมนุษย์
    return humanize_text(raw_text, sentiment)

# ==========================================
# 3. เริ่มสุ่มข้อมูล (รับประกันไม่ซ้ำ 100%)
# ==========================================
data = []
unique_comments = set()
start_date = datetime(2026, 1, 1)

while len(data) < NUM_RECORDS:
    rating = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.15, 0.40, 0.30])
    sentiment = 'positive' if rating >= 4 else 'negative' if rating <= 2 else random.choice(['positive', 'negative'])

    comment = generate_natural_comment(sentiment)

    # เคล็ดลับกันข้อมูลซ้ำ: ถ้าสุ่มแล้วดันไปซ้ำกับที่เคยมี ให้สุ่มเติม Spacebar หรือเครื่องหมาย ! เข้าไปนิดหน่อย
    while comment in unique_comments:
        comment += random.choice([" ", "!", ".", "คะ", "ครับ"])

    unique_comments.add(comment)

    # สุ่มตอบคำถาม Yes/No ให้สมจริงตามคะแนน
    prob_yes = 0.95 if sentiment == 'positive' else 0.20
    q_answers = np.random.choice(['Yes', 'No'], 8, p=[prob_yes, 1-prob_yes])

    ts = start_date + timedelta(days=random.randint(0, 90), hours=random.randint(8, 16), minutes=random.randint(0, 59))

    data.append({
        'feedback_id': f'F_IPD_{str(len(data)+1).zfill(5)}',
        'timestamp': ts.strftime('%Y-%m-%d %H:%M'),
        'q1_info_rights': q_answers[0],
        'q2_info_practice': q_answers[1],
        'q3_wristband_correct': q_answers[2],
        'q4_id_verification': q_answers[3],
        'q5_pain_fall_assess': q_answers[4],
        'q6_staff_intro': q_answers[5],
        'q7_decision_sharing': q_answers[6],
        'q8_discharge_plan': q_answers[7],
        'overall_rating': rating,
        'feedback_text': comment
    })

# ==========================================
# 4. บันทึกไฟล์
# ==========================================
df = pd.DataFrame(data)

print("\n--- ตัวอย่างข้อมูล IPD แบบภาษามนุษย์ (10 แถวแรก) ---")
for text in df['feedback_text'].head(10):
    print("-", text)

file_name = 'suth_hospital_ipd_natural_20k.csv'
df.to_csv(file_name, index=False, encoding='utf-8-sig')

process_time = time.time() - start_time
print(f"\n✅ สร้างข้อมูลสำเร็จ! {len(df):,} เรคคอร์ด")
print(f"📁 บันทึกไฟล์ '{file_name}' เรียบร้อย ใช้เวลาไป {process_time:.2f} วินาที")

กำลังสุ่มสร้างข้อมูล IPD ภาษามนุษย์จำนวน 500,000 เรคคอร์ด...

--- ตัวอย่างข้อมูล IPD แบบภาษามนุษย์ (10 แถวแรก) ---
- อยากให้ปรับปรุงด่วน พยาบาลบางคน เสียงดังและไม่ค่อยเย็น หมอมาตรวจแป๊บเดียวถามอาการไม่ทัน เหนื่อยใจจริงๆ
- ประทับใจม๊าก แม่บ้าน เอาใจใส่สุดๆ กดออดเรียกตอนดึกก็มาไวไม่เหวี่ยง ขอชื่นชมจากใจครับ
- อยากมาชื่นชม แม่บ้าน ดูแลดีดุจญาติมิตร เดินมาเช็คอาการตลอดเลย เป็นกำลังใจให้คนทำงานนะคะ คับ
- ขอบคุณทีมงานครับ พยาบาลวอร์ดนี้ ดูแลดีดุจญาติมิตร เดินมาเช็คอาการตลอดเลย เป็นกำลังใจให้คนทำงานนะคะ ค่ะ ❤️
- ดีเยี่ยมเลย พยาบาลวอร์ดนี้ เอาใจใส่สุดๆ ได้ห้องวิวดีพักผ่อนสบาย ขอชื่นชมจากใจครับ
- รู้สึกอบอุ่นใจ แม่บ้าน มือเบาม้ากกกตอนเจาะเลือด อาหารรพ.อร่อยกว่าที่คิดไว้เยอะ เป็นกำลังใจให้คนทำงานนะคะ
- อยากมาชื่นชม ทีมแพทย์ เอาใจใส่สุดๆ กดออดเรียกตอนดึกก็มาไวไม่เหวี่ยง เยี่ยมจริงๆ ค่ะ
- ดีเยี่ยมเลยจ้า ทีมแพทย์ ดูแลดีดุจญาติมิตร ได้ห้องวิวดีพักผ่อนสบาย ประทับใจสุดๆ
- อยากมาชื่นชม พยาบาลวอร์ดนี้ มือเบาม๊ากตอนเจาะเลือด ห้องพักสะอาดม๊ากแอร์เย็นฉ่ำ ❤️
- ดีเยี่ยมเลย เจ้าหน้าที่เวรดึก บริการรวดเร็วทันใ